In [4]:
import torch
from train import run_baseline_experiment, run_lora_experiment, run_lora_experiment_multiple_seeds
from utils import plot_results
print("CUDA available?", torch.cuda.is_available())
import lightning
print(lightning.__version__)
import numpy as np



CUDA available? True
2.5.0.post0


In [1]:
import torch
torch.cuda.empty_cache()

In [6]:
%%bash
python -c "import torch; print(torch.version.cuda)"
python -c "import torch; print(torch.cuda.device_count())"



11.8
2


In [7]:
for i in range(torch.cuda.device_count()):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")

Device 0: NVIDIA GeForce RTX 3090
Device 1: NVIDIA GeForce RTX 2080 Ti


## FPFT

In [ ]:
#TODO: rewrite all launches for the new API
#TODO: make them rerunnable (calculate acc over several runs with fixed randomstates)

In [8]:
# This either trains a new full‐parameter model on classes [0..4]
# or loads it from "base_model.ckpt" if found:
baseline_acc = run_baseline_experiment()
print("Baseline FPFT accuracy:", baseline_acc)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:15<00:00, 632kB/s] 


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 179kB/s]


...

Epoch 49: 100%|██████████| 220/220 [00:03<00:00, 72.80it/s, v_num=0, val_loss=0.0261, val_acc=0.991]


/home/sokoi0a/anaconda3/envs/rc_lora_env/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/checkpoint_connector.py:145: `.test(ckpt_path=None)` was called without a model. The best model of the previous `fit` call will be used. You can pass `.test(ckpt_path='best')` to use the best model or `.test(ckpt_path='last')` to use the last model. If you pass a value, this warning will be silenced.
Restoring states from the checkpoint path at logs/lightning_logs/version_0/checkpoints/epoch=49-step=11000.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at logs/lightning_logs/version_0/checkpoints/epoch=49-step=11000.ckpt


Testing DataLoader 0: 100%|██████████| 41/41 [00:00<00:00, 209.16it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.9945514798164368
        test_loss          0.019716547802090645
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 41/41 [00:00<00:00, 230.57it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.9945514798164368
        test_loss          0.019716545939445496
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Baseline FPFT accuracy: 0.9945514798164368


## LoRA

In [ ]:
# A:Gaussian; B:Zero
NUM_LAUNCHES = 20
seeds_to_try = np.arange(1,NUM_LAUNCHES+1)
average_acc = run_lora_experiment_multiple_seeds(seeds=seeds_to_try,rank=1, train_A=True, train_B=True, init_method_A='gaussian', init_method_B='zero', merge_frequency=0, use_stochastic=False)
print("LoRA (A:Gaussian; B:Zero):", average_acc)

In [ ]:
# A:Zero; B:Gaussian
NUM_LAUNCHES = 20
seeds_to_try = np.arange(1,NUM_LAUNCHES+1)
average_acc = run_lora_experiment_multiple_seeds(seeds=seeds_to_try, rank=1, train_A=True, train_B=True, init_method_A='zero', init_method_B='gaussian', init_method_B='zero', merge_frequency=0, use_stochastic=False)
print("LoRA (A:Zero; B:Gaussian):", average_acc)

## COLA

In [12]:
# A:Gaussian; B:Zero
acc = run_lora_experiment(rank=1, train_A=True, train_B=True, init_method_A='gaussian', init_method_B='zero', merge_frequency=1)
print("COLA (A:Gaussian; B:Zero):", acc)

/home/sokoi0a/_research_projects/2024_p-LORA-experiments/rac_lora_mnist/train.py:118: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("base_model.pt")


Epoch 49: 100%|██████████| 211/211 [00:03<00:00, 56.67it/s, v_num=3, val_loss=0.206, val_acc=0.931]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 211/211 [00:03<00:00, 56.55it/s, v_num=3, val_loss=0.206, val_acc=0.931]


/home/sokoi0a/anaconda3/envs/rc_lora_env/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/checkpoint_connector.py:145: `.test(ckpt_path=None)` was called without a model. The best model of the previous `fit` call will be used. You can pass `.test(ckpt_path='best')` to use the best model or `.test(ckpt_path='last')` to use the last model. If you pass a value, this warning will be silenced.
Restoring states from the checkpoint path at logs/lightning_logs/version_3/checkpoints/epoch=49-step=10550.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at logs/lightning_logs/version_3/checkpoints/epoch=49-step=10550.ckpt


Testing DataLoader 0: 100%|██████████| 38/38 [00:00<00:00, 164.20it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc             0.929232656955719
        test_loss           0.21218059957027435
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 38/38 [00:00<00:00, 173.02it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc             0.929232656955719
        test_loss           0.21218058466911316
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
COLA (A:Gaussian; B:Zero): 0.929232656955719


In [14]:
# A:Zero; B:Gaussian

acc = run_lora_experiment(
    rank=1,
    train_A=True,               # Train both A and B
    train_B=True,
    init_method_A='zero',   # A ~ Gaussian
    init_method_B='gaussian',       # B ~ Zero
    merge_frequency=1           # or 1; see note below
)
print("COLA (A:Zero; B:Gaussian):", acc)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type               | Params | Mode 
-------------------------------------------------------------
0 | val_accuracy  | MulticlassAccuracy | 0      | train
1 | test_accuracy | MulticlassAccuracy | 0      | train
2 | l1            | Linear             | 50.2 K | train
3 | l2            | Linear             | 4.2 K  | train
4 | l3            | Linear             | 325    | train
5 | dropout       | Dropout            | 0      | train
6 | relu          | ReLU               | 0      | train
  | other params  | n/a                | 1.0 K  | n/a  
-------------------------------------------------------------
1.0 K     Trainable params
54.7 K    Non-trainable params
55.8 K    Total params
0.223     Total estimated model params size (MB)
7         Modules in train mode
0         Modules in eval mode


Epoch 49: 100%|██████████| 211/211 [00:03<00:00, 57.03it/s, v_num=5, val_loss=0.133, val_acc=0.958]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 211/211 [00:03<00:00, 56.92it/s, v_num=5, val_loss=0.133, val_acc=0.958]


Restoring states from the checkpoint path at logs/lightning_logs/version_5/checkpoints/epoch=49-step=10550.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at logs/lightning_logs/version_5/checkpoints/epoch=49-step=10550.ckpt


Testing DataLoader 0: 100%|██████████| 38/38 [00:00<00:00, 190.90it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.9672906994819641
        test_loss           0.10157755017280579
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 38/38 [00:00<00:00, 194.24it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.9672906994819641
        test_loss           0.10157757252454758
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
COLA (A:Zero; B:Gaussian): 0.9672906994819641


In [5]:
# A:Gaussian; B:Gaussian
acc = run_lora_experiment(rank=1, train_A=True, train_B=True, init_method_A='gaussian', init_method_B='gaussian', merge_frequency=1)
print("COLA (A:Gaussian; B:Gaussian):", acc)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type               | Params | Mode 
-------------------------------------------------------------
0 | val_accuracy  | MulticlassAccuracy | 0      | train
1 | test_accuracy | MulticlassAccuracy | 0      | train
2 | l1            | Linear             | 50.2 K | train
3 | l2            | Linear             | 4.2 K  | train
4 | l3            | Linear             | 325    | train
5 | dropout       | Dropout            | 0      | train
6 | relu          | ReLU               | 0      | train
  | other params  | n/a                | 1.0 K  | n/a  
-------------------------------------------------------------
1.0 K     Trainable params
54.7 K    Non-trainable params
55.8 K    Total params
0.223     Total estimated model params size (MB)
7         Modules in train mode
0         Modules in eval mode


Epoch 10: 100%|██████████| 211/211 [00:03<00:00, 54.10it/s, v_num=11, val_loss=7.48e+3, val_acc=0.245]


Restoring states from the checkpoint path at logs/lightning_logs/version_11/checkpoints/epoch=10-step=2321.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at logs/lightning_logs/version_11/checkpoints/epoch=10-step=2321.ckpt


Testing DataLoader 0: 100%|██████████| 38/38 [00:00<00:00, 183.79it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.17774119973182678
        test_loss              16471.5546875
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 38/38 [00:00<00:00, 164.65it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.17774119973182678
        test_loss             16471.552734375
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
COLA (A:Gaussian; B:Gaussian): 0.17774119973182678


## AsymmLoRA

In [15]:
# A:Gaussian; B:Zero
acc = run_lora_experiment(rank=1, train_A=False, train_B=True, init_method_A='gaussian', init_method_B='zero', merge_frequency=0)
print("AsymmLoRA (A:Gaussian; B:Zero):", acc)

/home/sokoi0a/_research_projects/2024_p-LORA-experiments/rac_lora_mnist/train.py:118: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("base_model.pt")


Epoch 49: 100%|██████████| 211/211 [00:03<00:00, 56.99it/s, v_num=6, val_loss=1.050, val_acc=0.705]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 211/211 [00:03<00:00, 56.88it/s, v_num=6, val_loss=1.050, val_acc=0.705]


/home/sokoi0a/anaconda3/envs/rc_lora_env/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/checkpoint_connector.py:145: `.test(ckpt_path=None)` was called without a model. The best model of the previous `fit` call will be used. You can pass `.test(ckpt_path='best')` to use the best model or `.test(ckpt_path='last')` to use the last model. If you pass a value, this warning will be silenced.
Restoring states from the checkpoint path at logs/lightning_logs/version_6/checkpoints/epoch=49-step=10550.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at logs/lightning_logs/version_6/checkpoints/epoch=49-step=10550.ckpt


Testing DataLoader 0: 100%|██████████| 38/38 [00:00<00:00, 159.62it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.7329767346382141
        test_loss           1.0129774808883667
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 38/38 [00:00<00:00, 214.02it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.7329767346382141
        test_loss           1.0129773616790771
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
AsymmLoRA (A:Gaussian; B:Zero): 0.7329767346382141


In [16]:
# A:Zero; B:Gaussian
acc = run_lora_experiment(rank=1, train_A=True, train_B=False, init_method_A='zero', init_method_B='gaussian', merge_frequency=0)
print("AsymmLoRA (A:Gaussian; B:Zero):", acc)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type               | Params | Mode 
-------------------------------------------------------------
0 | val_accuracy  | MulticlassAccuracy | 0      | train
1 | test_accuracy | MulticlassAccuracy | 0      | train
2 | l1            | Linear             | 50.2 K | train
3 | l2            | Linear             | 4.2 K  | train
4 | l3            | Linear             | 325    | train
5 | dropout       | Dropout            | 0      | train
6 | relu          | ReLU               | 0      | train
  | other params  | n/a                | 1.0 K  | n/a  
-------------------------------------------------------------
912       Trainable params
54.9 K    Non-trainable params
55.8 K    Total params
0.223     Total estimated model params size (MB)
7         Modules in train mode
0         Modules in eval mode


Epoch 49: 100%|██████████| 211/211 [00:03<00:00, 56.60it/s, v_num=7, val_loss=1.370, val_acc=0.777]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 211/211 [00:03<00:00, 56.49it/s, v_num=7, val_loss=1.370, val_acc=0.777]


Restoring states from the checkpoint path at logs/lightning_logs/version_7/checkpoints/epoch=49-step=10550.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at logs/lightning_logs/version_7/checkpoints/epoch=49-step=10550.ckpt


Testing DataLoader 0: 100%|██████████| 38/38 [00:00<00:00, 139.28it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.7687718868255615
        test_loss           1.3863362073898315
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 38/38 [00:00<00:00, 176.83it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.7687718868255615
        test_loss            1.386336088180542
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
AsymmLoRA (A:Gaussian; B:Zero): 0.7687718868255615


## RAC-LoRA

In [ ]:
# A:Gaussian; B:Zero
acc = run_lora_experiment(rank=1, train_A=False, train_B=True, init_method_A='gaussian', init_method_B='zero', merge_frequency=1)
print("RAC-LoRA (A:Gaussian; B:Zero):", acc)

In [18]:
# A:Zero; B:Gaussian
acc = run_lora_experiment(rank=1, train_A=True, train_B=False, init_method_A='zero', init_method_B='gaussian', merge_frequency=1)
print("RAC-LoRA (A:Gaussian; B:Zero):", acc)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type               | Params | Mode 
-------------------------------------------------------------
0 | val_accuracy  | MulticlassAccuracy | 0      | train
1 | test_accuracy | MulticlassAccuracy | 0      | train
2 | l1            | Linear             | 50.2 K | train
3 | l2            | Linear             | 4.2 K  | train
4 | l3            | Linear             | 325    | train
5 | dropout       | Dropout            | 0      | train
6 | relu          | ReLU               | 0      | train
  | other params  | n/a                | 1.0 K  | n/a  
-------------------------------------------------------------
912       Trainable params
54.9 K    Non-trainable params
55.8 K    Total params
0.223     Total estimated model params size (MB)
7         Modules in train mode
0         Modules in eval mode


Epoch 49: 100%|██████████| 211/211 [00:03<00:00, 57.90it/s, v_num=9, val_loss=0.125, val_acc=0.963]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 211/211 [00:03<00:00, 57.79it/s, v_num=9, val_loss=0.125, val_acc=0.963]


Restoring states from the checkpoint path at logs/lightning_logs/version_9/checkpoints/epoch=49-step=10550.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at logs/lightning_logs/version_9/checkpoints/epoch=49-step=10550.ckpt


Testing DataLoader 0: 100%|██████████| 38/38 [00:00<00:00, 179.35it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.9613248109817505
        test_loss           0.11455155909061432
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing DataLoader 0: 100%|██████████| 38/38 [00:00<00:00, 189.34it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.9613248109817505
        test_loss           0.11455155164003372
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
RAC-LoRA (A:Gaussian; B:Zero): 0.9613248109817505


## Bernoulli-LoRA

In [ ]:
# in progress, one needs to experiments with prob and merge_frequency

acc = run_lora_experiment(rank=1,
    train_A=True,
    train_B=False,
    init_method_A='zero',
    init_method_B='gaussian',
    merge_frequency=1,         
    use_stochastic=True,       
    prob=0.5,
    deterministic_init=True,
    init_train_zero=True,     
    gaussian_resample=True
)
print("Final test acc =", acc)
